# Screencast 1 — Setup de SW (PyTorch)
### Módulo Edge AI — Processamento de IA Embarcado e Otimização de Modelos para Dispositivos de Borda

**Objetivo deste notebook:**
1. Preparar o ambiente de desenvolvimento (PyTorch, TorchVision, utilitários nativos de quantização/pruning do PyTorch).
2. Baixar e explorar o dataset de referência do módulo: **Casting Product Image Data for Quality Inspection** (classificação binária: peça `OK` vs. peça `Defeituosa`).
3. Construir um pipeline de dados (`torch.utils.data.Dataset` / `DataLoader`) reutilizável pelos próximos screencasts.
4. Treinar, via *transfer learning*, o **modelo de referência** deste módulo — que será quantizado (Screencast 2) e podado (Screencast 3).
5. Medir a linha de base de **tamanho, latência e acurácia** — a tríade de métricas que vamos acompanhar em todos os screencasts seguintes.


## 1. Ambiente de Desenvolvimento

Vamos instalar as peças centrais do pipeline de Edge AI em PyTorch:

- **Framework de treinamento:** PyTorch (`torch`) / TorchVision (modelos pré-treinados e transforms).
- **Biblioteca de conversão/exportação de modelo:** `torch.jit` (TorchScript) e `torch.onnx`, usadas para exportar o modelo para inferência em borda.
- **Ferramentas de otimização:** módulos nativos do PyTorch — `torch.ao.quantization` (quantização) e `torch.nn.utils.prune` (pruning) — usados nos Screencasts 2 e 3.

Se você já tem esse ambiente configurado, pode pular a célula de instalação e ir direto para a verificação de versões.

In [ ]:
# Execute esta célula apenas na primeira vez que configurar o ambiente.
# Recomenda-se um ambiente virtual (venv ou conda) dedicado a este módulo.

!pip install -r requirements.txt

In [ ]:
import os
import time
import json
import copy
import pathlib
import shutil
import zipfile

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import datasets, transforms, models

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDispositivo utilizado:", DEVICE)
print("(Sem problema se vier 'cpu' - este notebook roda bem em CPU.)")


### 2.1 Configurando as credenciais do Kaggle

O Kaggle emite um **API token** no novo formato (prefixo `KGAT_`):

1. Crie uma conta em [kaggle.com](https://www.kaggle.com) (se ainda não tiver).
2. Acesse **Account → Your API tokens → Create New Token** — a página mostra o token (`KGAT_...`) e um comando pronto.
3. No terminal (Linux/Mac), rode:
   ```bash
   mkdir -p ~/.kaggle && echo <SEU_TOKEN> > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token
   ```
   substituindo `<SEU_TOKEN>` pelo valor `KGAT_...` gerado (não cole o token em texto no notebook nem em nenhum arquivo versionado no git).
4. Alternativamente, você pode exportar a variável de ambiente `KAGGLE_API_TOKEN=<SEU_TOKEN>` antes de iniciar o Jupyter.


In [ ]:
KAGGLE_ACCESS_TOKEN = pathlib.Path.home() / ".kaggle" / "access_token"

kaggle_creds_ok = (
    KAGGLE_ACCESS_TOKEN.exists()
    or bool(os.environ.get("KAGGLE_API_TOKEN"))
)

if kaggle_creds_ok:
    if KAGGLE_ACCESS_TOKEN.exists():
        print(f"Credenciais do Kaggle encontradas em {KAGGLE_ACCESS_TOKEN}")
    else:
        print("Credenciais do Kaggle encontradas na variável de ambiente KAGGLE_API_TOKEN")
else:
    print(
        "Credenciais do Kaggle NAO encontradas.\n"
        "Siga as instrucoes da celula de texto acima antes de continuar,\n"
        "ou baixe o dataset manualmente em:\n"
        "https://www.kaggle.com/datasets/ravirajsinh45/real-life-industrial-dataset-of-casting-product\n"
        "e extraia o conteudo para a pasta ./data/casting_data/"
    )


In [ ]:
DATA_DIR = pathlib.Path("../data")
print(f"Arquivos de dados serão baixados em {DATA_DIR.resolve()}")
DATA_DIR.mkdir(exist_ok=True)

DATASET_SLUG = "ravirajsinh45/"
ZIP_PATH = DATA_DIR / "casting_data.zip"

if kaggle_creds_ok and not ZIP_PATH.exists():
    exit_code = os.system(
        f'kaggle datasets download -d {DATASET_SLUG} -p "{DATA_DIR}"'
    )
    if exit_code != 0:
        print("Falha ao baixar via Kaggle CLI. Baixe manualmente (ver célula acima).")
else:
    print("Zip já presente ou credenciais ausentes — pulando download automático.")


In [ ]:
# Extração do zip baixado pelo Kaggle CLI.
possible_zips = list(DATA_DIR.glob("*.zip"))
print("Zips encontrados em ./data:", possible_zips)

if possible_zips:
    zip_file = possible_zips[0]
    with zipfile.ZipFile(zip_file, "r") as zf:
        zf.extractall(DATA_DIR)
    print(f"Extraído {zip_file.name} para {DATA_DIR}/")
else:
    print("Nenhum zip encontrado - confirme se o dataset ja esta extraido em ./data/")


In [ ]:
# Localiza automaticamente as pastas train/ e test/ dentro do que foi extraído
# (a estrutura do zip do Kaggle costuma ser data/casting_data/casting_data/{train,test}/...)

def find_split_dirs(root: pathlib.Path):
    train_dirs = list(root.rglob("train"))
    test_dirs = list(root.rglob("test"))
    train_dir = next((d for d in train_dirs if (d / "ok_front").exists()), None)
    test_dir = next((d for d in test_dirs if (d / "ok_front").exists()), None)
    return train_dir, test_dir

TRAIN_DIR, TEST_DIR = find_split_dirs(DATA_DIR)
print("Pasta de treino:", TRAIN_DIR)
print("Pasta de teste :", TEST_DIR)

assert TRAIN_DIR is not None and TEST_DIR is not None, (
    "Nao encontrei train/ e test/ com as subpastas ok_front e def_front.\n"
    "Confira a extracao do dataset antes de continuar."
)


## 3. Explorando o Dataset

Antes de treinar qualquer modelo, vale confirmar visualmente o que estamos lidando: quantas imagens por classe, e como elas se parecem.


In [ ]:
def count_images(split_dir):
    counts = {}
    for class_dir in sorted(split_dir.iterdir()):
        if class_dir.is_dir():
            counts[class_dir.name] = len(list(class_dir.glob("*.jpeg"))) + len(list(class_dir.glob("*.jpg"))) + len(list(class_dir.glob("*.png")))
    return counts

train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR)

print("Treino:", train_counts, "| total:", sum(train_counts.values()))
print("Teste :", test_counts, "| total:", sum(test_counts.values()))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(6, 4))
axes[0].bar(train_counts.keys(), train_counts.values(), color=["#0000B0", "#008400"])
axes[0].set_title("Distribuicao - treino")
axes[1].bar(test_counts.keys(), test_counts.values(), color=["#0000B0", "#008400"])
axes[1].set_title("Distribuicao - teste")
plt.tight_layout()
plt.show()


In [ ]:
from PIL import Image

def show_samples(split_dir, n_per_class=4):
    classes = ["ok_front", "def_front"]
    fig, axes = plt.subplots(len(classes), n_per_class, figsize=(2.2 * n_per_class, 4.4))
    for row, cls in enumerate(classes):
        cls_dir = split_dir / cls
        files = list(cls_dir.glob("*.jpeg"))[:n_per_class]
        for col, f in enumerate(files):
            img = Image.open(f).convert("L")
            axes[row, col].imshow(img, cmap="gray")
            axes[row, col].axis("off")
            if col == 0:
                axes[row, col].set_ylabel(cls)
        axes[row, 0].set_title(cls, loc="left", fontsize=10)
    plt.tight_layout()
    plt.show()

show_samples(TRAIN_DIR)


## 4. Pipeline de Dados (`torch.utils.data`)

Padronizamos aqui o pipeline que os Screencasts 2 e 3 vão reutilizar:

- Redimensionamento para **224×224×3** (entrada padrão da ResNet50 no TorchVision).
- Conversão para 3 canais (o dataset é grayscale, mas os modelos pré-treinados esperam RGB).
- Normalização com a média/desvio-padrão do ImageNet.
- `datasets.ImageFolder` reconhece as subpastas `ok_front`/`def_front` como classes; remapeamos os índices para manter `ok_front -> 0` e `def_front -> 1`.
- Separação de 20% do conjunto de treino para validação, com seed fixa para reprodutibilidade.


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

torch.manual_seed(SEED)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

CLASS_TO_IDX = {"ok_front": 0, "def_front": 1}

def remap_target(default_class_to_idx):
    # ImageFolder ordena as classes alfabeticamente (def_front, ok_front);
    # este mapa converte esse índice para a convenção ok_front=0 / def_front=1.
    inverse_default = {v: k for k, v in default_class_to_idx.items()}
    return lambda idx: CLASS_TO_IDX[inverse_default[idx]]

full_train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
full_train_dataset.target_transform = remap_target(full_train_dataset.class_to_idx)

test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transform)
test_dataset.target_transform = remap_target(test_dataset.class_to_idx)

n_val = int(len(full_train_dataset) * 0.2)
n_train = len(full_train_dataset) - n_val
train_dataset, val_dataset = random_split(
    full_train_dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Treino: {n_train} imagens | Validação: {n_val} imagens | Teste: {len(test_dataset)} imagens")


## 5. Modelo de Referência — ResNet50 (Transfer Learning)

Estratégia:
1. Carregar a ResNet50 pré-treinada no ImageNet, via `torchvision.models`.
2. Congelar os pesos convolucionais (mantém o conhecimento visual já aprendido).
3. Substituir a camada `fc` original por uma cabeça de classificação binária simples.

O modelo produz um **logit** (sem sigmoid na saída); usamos `BCEWithLogitsLoss`, que combina sigmoid + binary cross-entropy de forma numericamente estável. A probabilidade é obtida aplicando `torch.sigmoid` sobre o logit apenas na hora de avaliar/prever.

Esse é o modelo que os próximos Screencasts.


In [ ]:
base_model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)

for param in base_model.parameters():
    param.requires_grad = False  # fase 1: só treinamos a cabeça de classificação

in_features = base_model.fc.in_features
base_model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, 128),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(128, 1),  # logit - sem ativação (BCEWithLogitsLoss aplica o sigmoid internamente)
)

model = base_model.to(DEVICE)
model.__class__.__name__  # apenas para referência; o modelo em si chama-se "resnet50" no torchvision

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parâmetros totais: {n_params:,} | Treináveis (fase 1): {n_trainable:,}")


## 6. Fine-Tuning

Duas fases, como é prática comum em transfer learning:

- **Fase 1:** treinar só a cabeça de classificação, com a ResNet50 congelada (rápido, poucas épocas).
- **Fase 2 (opcional):** descongelar o último bloco convolucional (`layer4`) e continuar o treinamento com uma taxa de aprendizado bem menor, para ajustar finamente os filtros ao domínio de peças fundidas.

Definimos abaixo um laço de treino/validação reutilizável, com *early stopping* (`patience=3`, restaurando os melhores pesos por `val_loss`).


In [ ]:
def run_epoch(loader, training):
    model.train(mode=training)
    total_loss, total_correct, total_samples = 0.0, 0, 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE)

            if training:
                optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            if training:
                loss.backward()
                optimizer.step()

            preds = (torch.sigmoid(logits) >= 0.5).float()
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)
            total_loss += loss.item() * labels.size(0)

    return total_loss / total_samples, total_correct / total_samples


def fit(epochs, patience=3):
    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(train_loader, training=True)
        val_loss, val_acc = run_epoch(val_loader, training=False)

        history["loss"].append(train_loss)
        history["accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)

        print(
            f"Epoch {epoch}/{epochs} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} "
            f"- val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping na época {epoch} (sem melhora em {patience} épocas).")
                break

    if best_state is not None:
        model.load_state_dict(best_state)  # restore_best_weights=True

    return history


history_head = fit(epochs=8, patience=3)


In [ ]:
# Fase 2 — fine-tuning do último bloco convolucional da ResNet50 (opcional)
for param in model.parameters():
    param.requires_grad = False
for param in model.layer4.parameters():  # último bloco convolucional (~equivalente às últimas 15 camadas)
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5,  # LR bem menor na fase 2
)

history_finetune = fit(epochs=5, patience=3)


In [ ]:
def plot_history(*histories):
    acc, val_acc, loss, val_loss = [], [], [], []
    for h in histories:
        acc += h["accuracy"]
        val_acc += h["val_accuracy"]
        loss += h["loss"]
        val_loss += h["val_loss"]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(acc, label="treino")
    axes[0].plot(val_acc, label="validação")
    axes[0].set_title("Acurácia")
    axes[0].legend()

    axes[1].plot(loss, label="treino")
    axes[1].plot(val_loss, label="validação")
    axes[1].set_title("Loss")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_history(history_head, history_finetune)


## 7. Avaliação do Modelo de Referência

Antes de salvar o modelo, vamos estabelecer a **linha de base** das três métricas que vão acompanhar todos os próximos screencasts: **tamanho em disco, latência de inferência e acurácia**.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
y_true_list, y_prob_list = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        logits = model(images)
        probs = torch.sigmoid(logits).cpu().numpy().ravel()
        y_prob_list.append(probs)
        y_true_list.append(labels.numpy().ravel())

y_true = np.concatenate(y_true_list, axis=0)
y_prob = np.concatenate(y_prob_list, axis=0)
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_true, y_pred, target_names=["ok_front", "def_front"]))
print("Matriz de confusao:\n", confusion_matrix(y_true, y_pred))

baseline_accuracy = float((y_pred == y_true).mean())
print(f"\nAcuracia no conjunto de teste: {baseline_accuracy:.4f}")


In [ ]:
def measure_latency_ms(eval_model, sample_batch, n_runs=100):
    eval_model.eval()
    sample_batch = sample_batch.to(DEVICE)
    with torch.no_grad():
        _ = eval_model(sample_batch)  # descarta a primeira execução (custo de warm-up)
        times = []
        for _ in range(n_runs):
            start = time.perf_counter()
            _ = eval_model(sample_batch)
            times.append((time.perf_counter() - start) * 1000)
    return float(np.mean(times)), float(np.std(times))

sample_batch, _ = next(iter(test_loader))
sample_batch = sample_batch[:1]  # latência para uma única imagem

mean_ms, std_ms = measure_latency_ms(model, sample_batch)
print(f"Latência média por imagem (CPU/GPU conforme DEVICE, PyTorch): {mean_ms:.2f} ms (+/- {std_ms:.2f} ms)")


In [ ]:
MODELS_DIR = pathlib.Path("../models")
MODELS_DIR.mkdir(exist_ok=True)

REFERENCE_MODEL_PATH = MODELS_DIR / "edge_ai_reference_resnet50_pytorch.pt"
model_cpu = model.to("cpu")
torch.save(model_cpu, REFERENCE_MODEL_PATH)
model.to(DEVICE)

baseline_size_mb = REFERENCE_MODEL_PATH.stat().st_size / (1024 * 1024)
print(f"Tamanho do modelo de referência em disco: {baseline_size_mb:.2f} MB")


## 8. Registrando a Linha de Base

Salvamos as métricas em um `baseline_metrics_pytorch.json` — os próximos Screencasts vão carregar este arquivo para comparar cada técnica de otimização contra a mesma referência.


In [ ]:
baseline_metrics = {
    "model_name": "edge_ai_reference_resnet50_pytorch",
    "framework": "pytorch",
    "size_mb": round(baseline_size_mb, 3),
    "latency_ms_mean": round(mean_ms, 3),
    "latency_ms_std": round(std_ms, 3),
    "test_accuracy": round(baseline_accuracy, 4),
    "img_size": list(IMG_SIZE),
    "device": str(DEVICE),
    "notes": "Modelo de referencia (ResNet50 + transfer learning, PyTorch) - linha de base para comparacao",
}

METRICS_PATH = MODELS_DIR / "baseline_metrics_pytorch.json"
with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(baseline_metrics, f, indent=2, ensure_ascii=False)

print(json.dumps(baseline_metrics, indent=2, ensure_ascii=False))
print(f"\nMétricas salvas em: {METRICS_PATH.resolve()}")


## 9. Checklist e Próximos Passos

- [x] Ambiente configurado (PyTorch, TorchVision).
- [x] Dataset de casting baixado e explorado.
- [x] Pipeline `torch.utils.data` padronizado (224×224, normalização ImageNet).
- [x] Modelo de referência (ResNet50 + transfer learning) treinado e avaliado.
- [x] Linha de base de tamanho, latência e acurácia salva em `models/baseline_metrics_pytorch.json`.
- [x] Modelo salvo em `models/edge_ai_reference_resnet50_pytorch.pt`.

**No Screencast 2 (Quantização),** vamos carregar exatamente esse arquivo `.pt` e aplicar:
- Quantização pós-treinamento dinâmica e estática (`torch.ao.quantization`).
- Quantização ciente do treinamento (*Quantization-Aware Training*, `torch.ao.quantization.quantize_qat`), caso a queda de acurácia da abordagem pós-treinamento seja alta demais.

E vamos comparar os resultados diretamente contra os números registrados aqui em `baseline_metrics_pytorch.json`.
